# 3단계: 재무 지표 계산
## 수익성 · 성장성 · 렌탈 특화 지표 직접 계산하기

---
### 이 실습에서 배우는 것
- 수익성 지표: OPM, NPM
- 성장성 지표: YoY, CAGR
- 렌탈 특화: ARPU, 계정 순증
- 공시 숫자 검증 (내 계산 vs 공시값 비교)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/coway_annual.csv')
df_q = pd.read_csv('data/coway_quarterly.csv')
df_os = pd.read_csv('data/coway_overseas.csv')

# 억원 단위로 통일
fin_cols = ['매출액', '영업이익', '법인세전이익', '당기순이익', '국내매출', '해외매출', '기타매출']
for col in fin_cols:
    df[col] = df[col] / 100

for col in ['매출액', '영업이익', '법인세전이익', '당기순이익']:
    df_q[col] = df_q[col] / 100

print('데이터 로드 완료')
df[['연도', '매출액', '영업이익', '당기순이익']].head()

## 3-1. 수익성 지표

In [ ]:
# 영업이익률 (OPM: Operating Profit Margin)
df['OPM'] = (df['영업이익'] / df['매출액'] * 100).round(1)

# 순이익률 (NPM: Net Profit Margin)
df['NPM'] = (df['당기순이익'] / df['매출액'] * 100).round(1)

# 법인세 부담률 (세금을 얼마나 내는가)
df['법인세부담률'] = ((df['법인세전이익'] - df['당기순이익']) / df['법인세전이익'] * 100).round(1)

print('=== 수익성 지표 ===')
print(df[['연도', '매출액', 'OPM', 'NPM', '법인세부담률']].to_string(index=False))

In [ ]:
# 분기별 OPM (계절성 확인)
df_q['OPM'] = (df_q['영업이익'] / df_q['매출액'] * 100).round(1)

# 4분기는 왜 영업이익률이 낮은가?
print('분기별 평균 OPM:')
print(df_q.groupby('분기번호')['OPM'].mean().round(1))
print('\n→ Q4가 낮다면 계절성, 매년 낮다면 구조적 문제')

## 3-2. 성장성 지표

In [ ]:
# YoY 성장률 (전년 대비)
df['매출_YoY'] = df['매출액'].pct_change() * 100
df['영업이익_YoY'] = df['영업이익'].pct_change() * 100
df['순이익_YoY'] = df['당기순이익'].pct_change() * 100

print('=== YoY 성장률 (%) ===')
print(df[['연도', '매출_YoY', '영업이익_YoY', '순이익_YoY']].round(1).to_string(index=False))

In [ ]:
# 공시 검증: IR 자료에 나온 숫자와 내 계산 비교
# 2025년 매출액: 연간 누계 공시 +15.2% YoY
공시값 = 15.2
내계산값 = df[df['연도'] == 2025]['매출_YoY'].values[0]

print(f'공시 발표값: +{공시값}%')
print(f'내 계산값:  +{내계산값:.1f}%')
print(f'차이: {abs(공시값 - 내계산값):.1f}%p')

if abs(공시값 - 내계산값) < 0.5:
    print('✅ 검증 통과 - 데이터가 정확합니다')
else:
    print('⚠️  차이 있음 - 데이터를 재확인하세요')

In [ ]:
# CAGR (연평균 성장률) 계산
def cagr(start_value, end_value, years):
    """CAGR = (종료값/시작값)^(1/연수) - 1"""
    return ((end_value / start_value) ** (1 / years) - 1) * 100

# 2022 → 2025 (3년)
매출_2022 = df[df['연도'] == 2022]['매출액'].values[0]
매출_2025 = df[df['연도'] == 2025]['매출액'].values[0]

영업이익_2022 = df[df['연도'] == 2022]['영업이익'].values[0]
영업이익_2025 = df[df['연도'] == 2025]['영업이익'].values[0]

순이익_2022 = df[df['연도'] == 2022]['당기순이익'].values[0]
순이익_2025 = df[df['연도'] == 2025]['당기순이익'].values[0]

print('=== 3년 CAGR (2022→2025) ===')
print(f'매출액 CAGR:   {cagr(매출_2022, 매출_2025, 3):.1f}%')
print(f'영업이익 CAGR: {cagr(영업이익_2022, 영업이익_2025, 3):.1f}%')
print(f'순이익 CAGR:   {cagr(순이익_2022, 순이익_2025, 3):.1f}%')
print()
print('→ 영업이익 CAGR > 매출 CAGR 이면 수익성 개선 중')
print('→ 순이익 CAGR > 영업이익 CAGR 이면 비영업 이익 기여 또는 세율 감소')

## 3-3. 렌탈 특화 지표 (코웨이 핵심)

In [ ]:
# ARPU (Average Revenue Per Unit - 계정당 평균 매출)
# 단위: 렌탈계정수 = 천 계정 → 만 단위 변환

df['렌탈계정수_만'] = df['렌탈계정수_천'] / 10

# ARPU = 연간 매출액(억원) / 계정수(만) = 억원/만계정
# → 만원/계정으로 변환: × 10000 / 10000 = ×1 (이미 만원 단위)
# 쉽게: 억원 × 1억 / (만계정 × 10000) = 만원/계정
df['ARPU_만원'] = (df['매출액'] * 10000 / (df['렌탈계정수_천'] * 10)).round(1)

# 계정 순증 (전년 대비 계정 증가수)
df['계정순증_천'] = df['렌탈계정수_천'].diff()
df['계정YoY'] = df['렌탈계정수_천'].pct_change() * 100

print('=== 렌탈 계정 지표 ===')
print(df[['연도', '렌탈계정수_만', '계정순증_천', '계정YoY', 'ARPU_만원']].round(1).to_string(index=False))
print()
print('→ 계정 순증이 플러스여야 미래 매출 성장 가능')
print('→ ARPU가 오르면 가격경쟁력 또는 프리미엄 제품 비중 증가')

## 3-4. 해외법인 성장성 분석

In [ ]:
# 법인별 YoY 성장률
df_os_pivot = df_os.pivot(index='연도', columns='법인', values='매출액_억원')
df_os_pivot.columns.name = None

df_os_yoy = df_os_pivot.pct_change() * 100

print('=== 해외법인별 매출 YoY 성장률 (%) ===')
print(df_os_yoy.round(1))

In [ ]:
# 미국 법인 흑자전환 시점 분석
df_us = df_os[df_os['법인'] == '미국'][['연도', '매출액_억원', '영업이익_억원']]
df_us['OPM'] = (df_us['영업이익_억원'] / df_us['매출액_억원'] * 100).round(1)

print('=== 미국 법인 실적 ===')
print(df_us.to_string(index=False))
print()

# 흑자전환 확인
흑자전환년도 = df_us[df_us['영업이익_억원'] > 0]['연도'].min()
if pd.notna(흑자전환년도):
    print(f'✅ 미국 법인 흑자전환: {int(흑자전환년도)}년')
else:
    print('⚠️  미국 법인 아직 적자 지속')

In [ ]:
# 해외 매출 비중 변화 (국내 포화 → 해외 성장 전략 검증)
df['해외비중'] = (df['해외매출'] / df['매출액'] * 100).round(1)
df['국내비중'] = (df['국내매출'] / df['매출액'] * 100).round(1)

print('=== 사업부문 매출 비중 ===')
print(df[['연도', '국내비중', '해외비중']].to_string(index=False))
print()
print('→ 해외 비중이 매년 증가하면 글로벌 확장 전략 성공 중')

---
## ✏️ 과제

1. 말레이시아 법인의 3년 CAGR을 계산해보세요.
2. 2025년 4Q 영업이익률(14.2%)이 연간 평균(17.7%)보다 낮은 이유를 데이터로 분석해보세요.
   - 힌트: 분기별 OPM 변동폭 확인
3. `ARPU_만원`이 증가하고 있다면 어떤 전략적 의미인지 해석해보세요.